# Exploración del Brent, el GPR y el mercado mundial

## 1. Objetivo

Explorar el dataset integrado y observar cómo se relacionan el precio Brent, el riesgo geopolítico y las variables mundiales de producción, consumo e inventarios.

## 2. Importación de librerías

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "processed" / "brent_gpr_market_monthly.csv"

## 3. Carga y verificación

Leemos el CSV integrado generado en el preprocesamiento y convertimos `month` a fecha.

In [ ]:
df = pd.read_csv(DATA_FILE)
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")

print("Dimensiones:", df.shape)
print("Período:", df["month"].min(), "a", df["month"].max())
print("Meses duplicados:", df["month"].duplicated().sum())
print("Valores faltantes:", int(df.isna().sum().sum()))
display(df.head())
display(df.tail())

## 4. Variables disponibles

| Variable | Descripción |
|---|---|
| `brent_price` | Precio promedio mensual del Brent. |
| `GPR`, `GPRT`, `GPRA` | Riesgo general, amenazas y actos geopolíticos. |
| `world_production` | Producción mundial, en millones de barriles diarios. |
| `world_consumption` | Consumo mundial, en millones de barriles diarios. |
| `world_inventory_net_withdrawals` | Retiros netos; un valor negativo indica acumulación. |
| `world_market_balance` | Producción menos consumo. |
| `brent_price_next_month` | Precio Brent observado en el mes siguiente. |

## 5. Estadísticas descriptivas

In [ ]:
df.describe().round(2)

## 6. Acontecimientos de referencia

Las fechas sirven como referencia visual. Una coincidencia temporal no demuestra que el acontecimiento haya causado un movimiento del precio.

In [ ]:
eventos = {
    "Atentados del 11-S": "2001-09-01",
    "Guerra de Irak": "2003-03-01",
    "Crisis Rusia-Ucrania": "2014-03-01",
    "Invasión rusa de Ucrania": "2022-02-01",
    "Invasión terrestre de Israel a Palestina": "2023-10-27",
    "Cierre del estrecho de Ormuz": "2026-03-01",
}
eventos = {nombre: pd.to_datetime(fecha) for nombre, fecha in eventos.items()}

def marcar_eventos(ax, etiquetas=True):
    for i, (nombre, fecha) in enumerate(eventos.items()):
        ax.axvline(fecha, color="black", linestyle="--", alpha=0.4)
        if etiquetas:
            ax.text(fecha, 0.98 if i % 2 == 0 else 0.70, nombre,
                    transform=ax.get_xaxis_transform(), rotation=90,
                    va="top", ha="right", fontsize=8)

## 7. Evolución del Brent y el GPR

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

sns.lineplot(data=df, x="month", y="brent_price", ax=axes[0], color="#1f77b4")
marcar_eventos(axes[0])
axes[0].set_title("Precio mensual del Brent")
axes[0].set_xlabel("")
axes[0].set_ylabel("USD por barril")

sns.lineplot(data=df, x="month", y="world_production", ax=axes[1], label="Producción")
sns.lineplot(data=df, x="month", y="world_consumption", ax=axes[1], label="Consumo")
marcar_eventos(axes[1], etiquetas=False)
axes[1].set_title("Producción y consumo mundial")
axes[1].set_xlabel("")
axes[1].set_ylabel("Millones de barriles por día")


sns.lineplot(data=df, x="month", y="GPR", ax=axes[2], color="#d62728")
marcar_eventos(axes[2], etiquetas=False)
axes[2].set_title("Índice GPR mensual")
axes[2].set_xlabel("Mes")
axes[2].set_ylabel("Índice GPR")

plt.tight_layout()
plt.show()

## 9. Inventarios y balance mundial

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
sns.lineplot(data=df, x="month", y="world_inventory_net_withdrawals", ax=axes[0], color="#e76f51")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Retiros netos de inventarios mundiales")
axes[0].set_ylabel("Millones de barriles por día")

sns.lineplot(data=df, x="month", y="world_market_balance", ax=axes[1], color="#2a9d8f")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Balance mundial: producción menos consumo")
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("Millones de barriles por día")

plt.tight_layout()
plt.show()

Las series presentan signos aproximadamente opuestos: cuando el consumo supera la producción, el mercado puede recurrir a retiros de inventarios.

## 10. Distribuciones de las variables mundiales

In [ ]:
market_columns = {
    "world_production": "Producción",
    "world_consumption": "Consumo",
    "world_inventory_net_withdrawals": "Retiros netos",
    "world_market_balance": "Balance mundial",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (column, label) in zip(axes.flat, market_columns.items()):
    sns.histplot(data=df, x=column, bins=25, kde=True, ax=ax)
    ax.set_title(f"Distribución de {label.lower()}")
    ax.set_xlabel(label)
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

Las distribuciones muestran el rango habitual y los meses más alejados. Esos valores deben analizarse antes de considerarlos errores.

## 11. Correlaciones entre niveles

Pearson resume asociaciones lineales; Spearman resume asociaciones monotónicas basadas en el orden de los valores.

In [ ]:
level_columns = [
    "brent_price", "GPR", "GPRT", "GPRA",
    "world_production", "world_consumption",
    "world_inventory_net_withdrawals", "world_market_balance",
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(df[level_columns].corr(method="pearson"), annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=axes[0])
axes[0].set_title("Correlación de Pearson")
sns.heatmap(df[level_columns].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=axes[1])
axes[1].set_title("Correlación de Spearman")
plt.tight_layout()
plt.show()

Las matrices permiten comparar relaciones lineales y monotónicas. Una correlación alta puede reflejar tendencias compartidas y no necesariamente una relación causal.

## 12. Variables actuales y Brent del mes siguiente

Aquí usamos `brent_price_next_month` porque es la variable que se buscará predecir. No es el precio del mismo mes: está desplazado una fila hacia el futuro.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (column, label) in zip(axes.flat, market_columns.items()):
    sns.scatterplot(data=df, x=column, y="brent_price_next_month", ax=ax, alpha=0.55)
    ax.set_title(f"{label} y Brent del mes siguiente")
    ax.set_xlabel(label)
    ax.set_ylabel("Brent del mes siguiente (USD)")

plt.tight_layout()
plt.show()

La dispersión muestra que ninguna variable mundial explica por sí sola el precio del mes siguiente.

## 13. Correlaciones entre variaciones

Las variaciones mensuales reducen el efecto de las tendencias de largo plazo y permiten comparar cambios ocurridos de un mes al siguiente.

In [ ]:
df["brent_variation_next_month"] = (
    df["brent_price_next_month"] / df["brent_price"] - 1
) * 100
df["GPR_variation"] = df["GPR"].diff()
df["production_variation_pct"] = df["world_production"].pct_change() * 100
df["consumption_variation_pct"] = df["world_consumption"].pct_change() * 100
df["inventory_variation"] = df["world_inventory_net_withdrawals"].diff()
df["balance_variation"] = df["world_market_balance"].diff()

variation_columns = [
    "brent_variation_next_month", "GPR_variation",
    "production_variation_pct", "consumption_variation_pct",
    "inventory_variation", "balance_variation",
]

plt.figure(figsize=(10, 7))
sns.heatmap(df[variation_columns].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Correlación de Spearman entre variaciones")
plt.tight_layout()
plt.show()

## 14. Conclusiones iniciales

- El dataset integrado permite estudiar conjuntamente el Brent, el GPR y el mercado mundial.
- Producción y consumo presentan tendencias compartidas, mientras que inventarios y balance muestran cambios mensuales más marcados.
- Las correlaciones de niveles deben interpretarse con cautela porque varias series cambian con el tiempo.
- Las correlaciones y dispersiones describen asociaciones, no causalidad.
